# Décideur de triage — ModernCamemBERT-bio-v2-base

Entraînement des deux variantes (complet, patient seul) sur le dataset `data/decideur`.

Prérequis : accélérateur **GPU** activé, **Internet** activé, dataset Kaggle contenant `complet/`, `patient/` et `train_decideur.py` ajouté via *Add Input*.

In [1]:
# 1. Vérifier le GPU et les versions
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__, '| CUDA', torch.cuda.is_available())

name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB
torch 2.10.0+cu128 | transformers 5.0.0 | CUDA True


In [2]:
# 2. Repérer les fichiers du dataset
!find /kaggle/input -maxdepth 4 -name '*.jsonl' -o -maxdepth 4 -name 'train_decideur.py'

find: warning: you have specified the global option -maxdepth after the argument -name, but global options are not positional, i.e., -maxdepth affects tests specified before it as well as those specified after it.  Please specify global options before other arguments.
/kaggle/input/datasets/veroniqueleroux/train-decideur/train_decideur.py
/kaggle/input/datasets/veroniqueleroux/patient/validation.jsonl
/kaggle/input/datasets/veroniqueleroux/patient/train.jsonl
/kaggle/input/datasets/veroniqueleroux/complet/validation.jsonl
/kaggle/input/datasets/veroniqueleroux/complet/train.jsonl


In [3]:
# 3. Chemins (déduits automatiquement ; à corriger si l'affichage ci-dessous est faux)
from pathlib import Path
SCRIPT = next(Path('/kaggle/input').rglob('train_decideur.py'))
DATA = next(Path('/kaggle/input').rglob('complet/train.jsonl')).parent.parent
print('Script :', SCRIPT)
print('Données :', DATA, '->', sorted(p.name for p in DATA.iterdir()))

Script : /kaggle/input/datasets/veroniqueleroux/train-decideur/train_decideur.py
Données : /kaggle/input/datasets/veroniqueleroux -> ['complet', 'patient', 'train-decideur']


In [4]:
# 4. Run 1 : conversation complète
!CUDA_VISIBLE_DEVICES=0 python {SCRIPT} --data-dir {DATA}/complet --output-dir /kaggle/working/decideur-complet

Generating train split: 907 examples [00:00, 17164.10 examples/s]
Generating validation split: 107 examples [00:00, 27310.32 examples/s]
config.json: 1.13kB [00:00, 378kB/s]
tokenizer_config.json: 1.33kB [00:00, 3.69MB/s]
tokenizer.json: 756kB [00:00, 24.5MB/s]
Map: 100%|███████████████████████████| 107/107 [00:00<00:00, 1278.41 examples/s]
[train] 907 exemples, 0 tronqués à 768 tokens
[validation] 107 exemples, 0 tronqués à 768 tokens
model.safetensors: 100%|██████████████████████| 645M/645M [00:03<00:00, 167MB/s]
Loading weights: 100%|█| 136/136 [00:00<00:00, 1825.26it/s, Materializing param=
ModernBertForSequenceClassification LOAD REPORT from: almanach/ModernCamemBERT-bio-v2-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
decoder.weight    | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expec

In [5]:
# 5. Run 2 : messages du patient uniquement
!CUDA_VISIBLE_DEVICES=0 python {SCRIPT} --data-dir {DATA}/patient --output-dir /kaggle/working/decideur-patient

Generating train split: 907 examples [00:00, 107619.27 examples/s]
Generating validation split: 107 examples [00:00, 26655.02 examples/s]
Map: 100%|███████████████████████████| 107/107 [00:00<00:00, 1397.53 examples/s]
[train] 907 exemples, 0 tronqués à 768 tokens
[validation] 107 exemples, 0 tronqués à 768 tokens
Loading weights: 100%|█| 136/136 [00:00<00:00, 1488.16it/s, Materializing param=
ModernBertForSequenceClassification LOAD REPORT from: almanach/ModernCamemBERT-bio-v2-base
Key               | Status     | 
------------------+------------+-
decoder.weight    | UNEXPECTED | 
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use 

In [6]:
# 6. Résumé côte à côte
import json
for v in ['complet', 'patient']:
    r = json.load(open(f'/kaggle/working/decideur-{v}/resultats_validation.json'))
    print(f'\n===== {v}')
    for regle in [k for k in r if k != 'parametres']:
        m = r[regle]
        print(f"{regle:<28} exact. {m['exactitude']:.3f} | F1 {m['f1_macro']:.3f} | sous-triage {m['taux_sous_triage']:.3f} | rappel urg. {m['rappel_urgence_maximale']:.3f} | rappel diff. {m['rappel_differee']:.3f}")


===== complet
argmax                       exact. 0.841 | F1 0.829 | sous-triage 0.065 | rappel urg. 0.978 | rappel diff. 0.862
seuil_0.3                    exact. 0.841 | F1 0.829 | sous-triage 0.065 | rappel urg. 0.978 | rappel diff. 0.862
reference_toujours_urgence   exact. 0.421 | F1 0.197 | sous-triage 0.000 | rappel urg. 1.000 | rappel diff. 0.000

===== patient
argmax                       exact. 0.832 | F1 0.820 | sous-triage 0.075 | rappel urg. 0.933 | rappel diff. 0.828
seuil_0.3                    exact. 0.860 | F1 0.845 | sous-triage 0.047 | rappel urg. 1.000 | rappel diff. 0.828
reference_toujours_urgence   exact. 0.421 | F1 0.197 | sous-triage 0.000 | rappel urg. 1.000 | rappel diff. 0.000


In [7]:
# 7. Archive à télécharger (panneau Output, à droite)
!cd /kaggle/working && rm -rf decideur-*/checkpoints && zip -qr decideur_resultats.zip decideur-complet decideur-patient && ls -lh decideur_resultats.zip

-rw-r--r-- 1 root root 965M Oct  2 14:59 decideur_resultats.zip
